# Guardian Lab AI — Frozen Logistic Regression v6

Notebook utama memuat artifact beku dan laporan v6. Tidak melakukan training, memilih threshold, mengunjungi URL atau memerlukan model lain. URL hasil dekode QR memakai model yang sama. Status tetap riset nonkomersial/internal shadow; LOW tidak menjamin aman.

In [1]:
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(
    p
    for p in (Path.cwd(), Path.cwd().parent)
    if (p / "ml/policies/shadow-runtime-manifest.json").is_file()
).resolve()
sys.path.insert(0, str(ROOT / "ml/src"))

In [2]:
from guardian_ml.paths import ROOT as PACKAGE_ROOT
from guardian_ml.rich_artifact import RichArtifact
from guardian_ml.preprocessing import prepare_url
from guardian_ml.shadow_predictor import ShadowPredictor
from guardian_ml.release_gates import assess_release

assert PACKAGE_ROOT == ROOT
manifest = json.loads((ROOT / "ml/policies/shadow-runtime-manifest.json").read_text())
report = json.loads((ROOT / "ml/reports/data-improvement-v6.json").read_text())
assert report["bundle_sha256"] == manifest["bundle_sha256"]
assert report["thresholds"] == manifest["thresholds"]
artifact_path = ROOT / manifest["artifact_path"]
frozen_path = ROOT / manifest["frozen_metadata_path"]
artifact = RichArtifact() if artifact_path.is_file() and frozen_path.is_file() else None
print(
    {
        "model": manifest["model_version"],
        "artifact_available": artifact is not None,
        "fit_rows": report["fit_rows"],
        "thresholds": manifest["thresholds"],
        "training_performed": False,
        "production_ready": False,
    }
)

{'model': 'notebook-url-core-v6', 'artifact_available': True, 'fit_rows': 532793, 'thresholds': {'warning': 0.904713285827399, 'high': 0.9884744926236343}, 'training_performed': False, 'production_ready': False}


## Evaluasi asli v6

Warning mencakup MEDIUM+HIGH pada threshold asli. Holdout vendor domain-disjoint berasal dari sumber yang juga dipakai development. Dataset berawalan observed telah teramati; audit manusia kecil bukan gold Indonesia. PhishTank positif saja tidak memiliki denominator benign.

In [3]:
rows = []
for name, benchmark in report["binary_benchmarks"].items():
    result = benchmark["v6"]["warning"]
    rows.append(
        {
            "benchmark": name,
            **{
                key: result[key]
                for key in ("rows", "accuracy", "precision", "recall", "fpr")
            },
        }
    )
positive = report["new_community_positive_sensitivity"]["v6"]["warning"]
rows.append(
    {
        "benchmark": "phishtank_2026_positive_only",
        "rows": positive["rows"],
        "accuracy": None,
        "precision": None,
        "recall": positive["recall"],
        "fpr": None,
    }
)
display(pd.DataFrame(rows).set_index("benchmark"))
display(
    pd.DataFrame(
        report["binary_benchmarks"]["new_deepurlbench_domain_holdout"]["v6"]["warning"][
            "confusion_matrix"
        ],
        index=["vendor warning"],
    )
)
print(
    "Batas ketidakpastian vendor:",
    report["binary_benchmarks"]["new_deepurlbench_domain_holdout"]["v6"]["warning"][
        "fpr_wilson95_rows_iid_assumption"
    ],
)
print(
    "Feed positif hanya mengukur recall; benchmark precision bukan precision trafik Indonesia."
)

,rows,accuracy,precision,recall,fpr
benchmark,,,,,
new_deepurlbench_domain_holdout,20000,0.932250,0.859060,0.463139,0.009444
observed_v5_deepurlbench,50000,0.928720,0.850278,0.455756,0.010340
observed_v5_human_audit,33,0.454545,1.000000,0.181818,0.000000
observed_phresh_novel_domain,80052,0.637548,0.997862,0.430951,0.001613
observed_group_holdout_mendeley2020,1002,0.792415,0.997967,0.703438,0.003289
observed_group_holdout_phishvn_core,2123,0.930287,0.920000,0.396552,0.004231
observed_group_holdout_phiusiil,20219,0.736980,0.998991,0.358402,0.000251
observed_urlphish,85824,0.968167,0.978068,0.553877,0.000927
phishtank_2026_positive_only,9269,NaN,NaN,0.465314,NaN


,tn,fp,fn,tp
vendor warning,17621,168,1187,1024


Batas ketidakpastian vendor: [0.008124845322761243, 0.010975052779923177]
Feed positif hanya mengukur recall; benchmark precision bukan precision trafik Indonesia.


## Integritas dan parity

Referensi berikut memakai input sintetis, dibuat dari fungsi notebook training v6 asli sebelum cleanup. Tidak ada baris dataset yang ditampilkan. Tanpa artifact lokal, perhitungan prediksi tidak tersedia; tidak diganti skor LOW.

In [4]:
if artifact is None:
    print({"status": "unavailable", "model_score": None, "risk_band": None})
else:
    reference = json.loads(
        (ROOT / "ml/provenance/v6-runtime-reference.json").read_text()
    )
    assert reference["artifact_sha256"] == artifact.manifest["bundle_sha256"]
    assert (
        reference["training_notebook_code_sha256"] == artifact.metadata["code_sha256"]
    )
    cores = [prepare_url(case["url"]).core for case in reference["cases"]]
    assert cores == [case["core"] for case in reference["cases"]]
    expected = np.array([case["probability"] for case in reference["cases"]])
    delta = float(np.max(np.abs(artifact.predict_cores(cores) - expected)))
    assert delta < reference["absolute_tolerance"]
    predictor = ShadowPredictor(artifact)
    direct = predictor.predict("https://example.org/login", "URL")
    qr = predictor.predict("https://example.org/login", "QR")
    assert direct.model_score == qr.model_score
    assert not direct.production_ready and direct.needs_further_review
    print(
        {
            "synthetic_cases": len(cores),
            "maximum_probability_delta": delta,
            "url_qr_parity": True,
        }
    )

{'synthetic_cases': 62, 'maximum_probability_delta': 0.0, 'url_qr_parity': True}


In [5]:
gate = assess_release()
assert not gate["production_ready"]
display(pd.DataFrame(gate["findings"]))
print(
    {
        "status": gate["status"],
        "pending_evidence": gate["pending_evidence"],
        "production_ready": False,
    }
)

,criterion,status,observed,target
0,vendor_recall_lower95,failed,0.442436,0.85
1,vendor_fpr_upper95,failed,0.010975,0.01
2,community_recall_lower95,failed,0.455177,0.85


{'status': 'not_ready', 'pending_evidence': ['independent_indonesian_human_gold_set', 'two_human_reviews_privacy_provenance_and_adjudication', 'campaign_time_segment_and_regression_review', 'contemporary_benign_traffic_evidence'], 'production_ready': False}


## Batas kemampuan

Recall phishing masih rendah dan sebagian benchmark mengalami regresi terhadap baseline historis. Perbandingan asli dan kontrol yang dideklarasikan terlambat tetap ada dalam laporan agregat. Query/fragment dikeluarkan; ancaman parameter-only dapat terlewat. Sanitizer regex memiliki gap untuk secret pendek dalam assignment path. Belum ada gold Indonesia, training baru atau promosi produksi.

Identitas dan provenance tersedia di `ml/provenance/v6-origin.json`; protokol dan laporan lengkap ada di `ai-agent-docs/data-improvement-v6-plan.md` dan `ml/reports/data-improvement-v6.json`. Replay lokal dua holdout memakai `scripts/validate_v6.py --evaluate-local`, setelah memasang extra `evaluation`.